In [1]:
1+1

2

In [1]:
!pip install torch-geometric-signed-directed

  Using cached torch_geometric_signed_directed-1.1.1-py3-none-any.whl.metadata (27 kB)
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
  Using cached xxhash-3.7.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
Using cached torch_geometric_signed_directed-1.1.1-py3-none-any.whl (119 kB)
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)
Using cached xxhash-3.7.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (193 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torch-geometric-signed-directed]ic]


In the following code snippets, we overview a simple end-to-end machine learning pipeline designed with PyTorch Geometric Signed Directed for directed networks. These code snippets solve a link direction prediction problem on a real-world data set. The pipeline consists of data preparation, model definition, training, and evaluation phases.

In [8]:
# !pip install numpy

In [9]:
# !pip install scikit-learn

In [10]:
from sklearn.metrics import accuracy_score
import torch

from torch_geometric_signed_directed.utils import link_class_split, in_out_degree
from torch_geometric_signed_directed.nn.directed import MagNet_link_prediction
from torch_geometric_signed_directed.data import load_directed_real_data

device = torch.device('cuda' if \
torch.cuda.is_available() else 'cpu')

In [11]:
load_directed_real_data?

In [12]:
data = load_directed_real_data(dataset='webkb', root='./', name='cornell').to(device)
link_data = link_class_split(data, prob_val=0.15, prob_test=0.05, task = 'direction', device=device)

Processing...
Done!



First of all, after importing and defining the device, we load the DirectedData object for the selected data set and map it to the device. We then create a train-validation-test split of the edge set by using the directed link splitting function.

In [13]:
model = MagNet_link_prediction(q=0.25, K=1, num_features=2, hidden=16, label_dim=2).to(device)
criterion = torch.nn.NLLLoss()


In the second snippet, we first construct the model instance, then initialize the cross-entropy loss function.

In [14]:
def train(X_real, X_img, y, edge_index, edge_weight, query_edges):
    model.train()
    out = model(X_real, X_img, edge_index=edge_index,
                    query_edges=query_edges,
                    edge_weight=edge_weight)
    loss = criterion(out, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_acc = accuracy_score(y.cpu(),
    out.max(dim=1)[1].cpu())
    return loss.detach().item(), train_acc

def test(X_real, X_img, y, edge_index, edge_weight, query_edges):
    model.eval()
    with torch.no_grad():
        out = model(X_real, X_img, edge_index=edge_index,
                    query_edges=query_edges,
                    edge_weight=edge_weight)
    test_acc = accuracy_score(y.cpu(),
    out.max(dim=1)[1].cpu())
    return test_acc


In the third part, we define the training and evaluation functions. Setting the model to be trainable, we obtain edge class assignment probablities with a forward pass of the model instance. We then obtain the training loss value. After that, we backpropagate and update the model parameters. Then, we calculate the accuracy of the training samples. Finally, we return the loss value as well as the training accuracy.

For the evaluation function (named test), we do not set the model to be trainable. With a forward pass, we obtain the probability assignment matrix. We then obtain test accuracy and return the result.

In [15]:

for split in list(link_data.keys()):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01,
    weight_decay=0.0005)
    edge_index = link_data[split]['graph']
    edge_weight = link_data[split]['weights']
    query_edges = link_data[split]['train']['edges']
    y = link_data[split]['train']['label']
    X_real = in_out_degree(edge_index,
    size=len(data.x)).to(device)
    X_img = X_real.clone()
    query_val_edges = link_data[split]['val']['edges']
    y_val = link_data[split]['val']['label']
    for epoch in range(200):
        train_loss, train_acc = train(X_real,
        X_img, y, edge_index, edge_weight, query_edges)
        val_acc = test(X_real, X_img, y_val,
        edge_index, edge_weight, query_val_edges)
        print(f'Split: {split:02d}, Epoch: {epoch:03d}, \
        Train_Loss: {train_loss:.4f}, Train_Acc: \
        {train_acc:.4f}, Val_Acc: {val_acc:.4f}')

    query_test_edges = link_data[split]['test']['edges']
    y_test = link_data[split]['test']['label']
    test_acc = test(X_real, X_img, y_test, edge_index,
    edge_weight, query_test_edges)
    print(f'Split: {split:02d}, Test_Acc: {test_acc:.4f}')
    model.reset_parameters()


Split: 00, Epoch: 000,         Train_Loss: 6.9130, Train_Acc:         0.4567, Val_Acc: 0.5897
Split: 00, Epoch: 001,         Train_Loss: 3.2204, Train_Acc:         0.5817, Val_Acc: 0.6282
Split: 00, Epoch: 002,         Train_Loss: 1.4551, Train_Acc:         0.7500, Val_Acc: 0.6923
Split: 00, Epoch: 003,         Train_Loss: 0.5498, Train_Acc:         0.8486, Val_Acc: 0.6923
Split: 00, Epoch: 004,         Train_Loss: 0.3073, Train_Acc:         0.8894, Val_Acc: 0.7051
Split: 00, Epoch: 005,         Train_Loss: 0.2798, Train_Acc:         0.8990, Val_Acc: 0.6923
Split: 00, Epoch: 006,         Train_Loss: 0.2758, Train_Acc:         0.9062, Val_Acc: 0.6923
Split: 00, Epoch: 007,         Train_Loss: 0.2881, Train_Acc:         0.9111, Val_Acc: 0.7051
Split: 00, Epoch: 008,         Train_Loss: 0.2580, Train_Acc:         0.9159, Val_Acc: 0.7051
Split: 00, Epoch: 009,         Train_Loss: 0.2425, Train_Acc:         0.9207, Val_Acc: 0.7051
Split: 00, Epoch: 010,         Train_Loss: 0.2389, Train_Acc

KeyboardInterrupt: 

We run the actual experiments in the last code snippet. For each of the data splits, we first initialize the optimizer. We then prepare data objects to be used, and start the training process. For each epoch, we apply the training function to obtain training loss and accuracy, then evaluate with the test() function on validation nodes. We then print the training and validation results. After training, we prepare test data, obtain the test performance, and print some logs. Finally, we reset model parameters and iterate to the next data split loop.

